In [12]:
# tools -> duckduckgo, yfinance, llm, prompt template
# https://open-meteo.com/en/docs
import requests
import yfinance as yf
from langsmith import Client

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate    
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents.react.agent import create_react_agent


In [13]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

llm.invoke("hi").content

'Hello! How can I help you today?'

In [14]:
search_tool = DuckDuckGoSearchRun()

search_tool.invoke("gujarati news")

"Janmabhoomi is a Gujarati-language evening daily newspaper based in Mumbai, India, founded on 9 June 1934 by freedom fighter Amritlal Sheth through the Saurashtra Trust. The publication emerged during India's independence movement as part of the Sta… Gujarati News Samachar - Find all Gujarati News and Samachar, News in Gujarati, Gujarat News, Gujarati News Headlines and Daily Breaking News, Gujarati News Paper in DivyaBhaskar.co.in. Latest News in Gujarati: Get real-time breaking news and top headlines from Gujarat, including updates from Ahmedabad, Rajkot, Vadodara, Bhavnagar, Gandhinagar, Banaskantha, along with politics, sports, entertainment, and more only on News18 ગુજરાતી. 23 minutes ago · ABP Asmita Gujarati News: ગુજરાતી ન્યૂઝ, Breaking News in Gujarati, Top Headlines in Gujarati (ગુજરાતીમાં ટોપ હેડલાઈન્સ), Gujarati Latest News (ગુજરાતી લેટેસ્ટ ન્યૂઝ), Today Top News (આજના ટોપ ન્યૂઝ), Gujarati News ... Sandesh is one of the leading Gujarati News paper. Get all latest news, ગુજ

In [15]:
@tool
def get_weather(city: str) -> str: 
    '''Get the currrent wether of the given city name'''

    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={'name': city, 'count': 1}
    ).json()
    
    if not geo.get("results"):
        return f"Could not find location: {city}"

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": True},
    ).json()

    current = weather["current_weather"]
    return f"{city}: {current['temperature']}°C, wind {current['windspeed']} km/h"

    
get_weather.invoke({"city":'ahmedabad'})

'ahmedabad: 26.4°C, wind 6.9 km/h'

In [16]:
@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol, e.g. AAPL, TSLA, INFY.NS."""
    stock = yf.Ticker(ticker)
    history = stock.history(period="1d")
    
    if history.empty:
        return f"Could not retrieve stock price for ticker: {ticker}"
        
    price = history["Close"].iloc[-1]
    return f"{ticker.upper()} current price: ${price:.2f}"

get_stock_price.invoke({"ticker": "TSLA"})

'TSLA current price: $365.44'

In [17]:
llm_with_tools = llm.bind_tools([search_tool, get_weather, get_stock_price])

response = llm_with_tools.invoke('what is the weather of ahmedabad today ? and what is current nifty stock price ?  ')
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get weather for Ahmedabad and current Nifty stock price. Use get_weather and get_stock_price (ticker for Nifty? NIFTY 50 maybe ticker "^NSEI"? Not sure. Could use "NIFTY" maybe. Let\'s try get_stock_price with ticker "NIFTY".', 'tool_calls': [{'id': 'fc_ce64bc06-1825-423d-8f2c-7ea5c0ee3d07', 'function': {'arguments': '{"city":"Ahmedabad"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 236, 'total_tokens': 328, 'completion_time': 0.203013973, 'completion_tokens_details': {'reasoning_tokens': 64}, 'prompt_time': 0.075854186, 'prompt_tokens_details': None, 'queue_time': 0.414508133, 'total_time': 0.278868159}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09f25-69db-7fd0-b116-ed7dea34a638-0'

In [18]:
tools = [search_tool, get_weather, get_stock_price]

client = Client()
react_prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True,
)
print(react_prompt.template)

agent = create_react_agent(llm, tools, react_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,             # Thought
    handle_parsing_errors=True,  # recovers if the model's text doesn't match the expected format
)

result = agent_executor.invoke({
    "input": "What's the weather in Ahmedabad and TSLA's current stock price?"
})
print(result["output"])

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


> Entering new AgentExecutor chain...
Question: What's the weather in Ahmedabad and TSLA's current stock price?
Thought: I need to retrieve the current weather for Ahmedabad and the current stock price for Tesla (ticker TSLA). I will use the provided tools: get_weather for the weather and get_stock_price for the stock price.
Action: get_weather
Action Input: AhmedabadAhmedabad: 26.4°C, wind 6.9 km/hInvalid Format: M